# B-03. 고전 모델 전체 + 특성 ablation

모델 5종(로지스틱 L2·L1, KNN, 랜덤포레스트, 특성 MLP) × 특성 조합 5종을 녹화 하나 빼기 21-fold로 비교한다.

| 특성 조합 | 내용 |
|---|---|
| A | 진동 세기 (표준편차, RMS, 최댓값−최솟값) |
| AB | A + 분포 형태 (평균, 왜도, 첨도) |
| ABC | AB + 주파수 대역 에너지 |
| AC | B 없이 A + C |
| ABC-mean | ABC에서 채널 평균 6개만 뺌 → "평균이 녹화를 외우는 단서인가" 확인 |

런타임은 CPU로 충분 (전체 10~15분)

In [ ]:
# 1. 레포 받기 + 데이터
import os
if not os.path.exists('/content/ML-imu'):
    !git clone -q https://github.com/ekgP525/ML-imu.git /content/ML-imu
%cd /content/ML-imu
!git pull -q
!pip install -q -r requirements.txt
!python data/download.py

In [ ]:
# 2. 불러오기
import sys, time, warnings
sys.path += ['src', 'notebooks/B_classic']
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt

from recordings import load_dataset
from features import extract_features
from evaluate import evaluate
from b_classic import make_classic, select_features, run_loro

X, y, rec = load_dataset('data')
F_all = extract_features(X, 'ABC')
print(X.shape, F_all.shape)

In [ ]:
# 3. 모델 × 특성 조합 전부 실행 (21-fold)
MODELS = ['logreg_l2', 'logreg_l1', 'knn', 'rf', 'mlp']
SPECS = ['A', 'AB', 'ABC', 'AC', 'ABC-mean']
rows, preds = [], {}
for spec in SPECS:
    F = select_features(F_all, spec)
    for name in MODELS:
        t = time.time()
        p = run_loro(lambda: make_classic(name), F, y, rec)
        m = evaluate(y, p)
        preds[f'{name}|{spec}'] = p
        rows.append({'model': name, 'features': spec, **m})
        print(f"{spec:9s} {name:10s} macro-F1 {m['macro_f1']:.3f}  acc {m['accuracy']:.3f}  "
              f"mode {m['mode_macro_f1']:.3f}  ({time.time() - t:.0f}s)")
res = pd.DataFrame(rows)

In [ ]:
# 4. 표: macro-F1 (행 = 모델, 열 = 특성 조합). 규칙 기준선 0.351
print('macro-F1'); display(res.pivot(index='model', columns='features', values='macro_f1')[SPECS].round(3))
print('모드 macro-F1 (규칙 기준선 0.666)'); display(res.pivot(index='model', columns='features', values='mode_macro_f1')[SPECS].round(3))

In [ ]:
# 5. 그림: 특성 조합별 macro-F1
tab = res.pivot(index='features', columns='model', values='macro_f1').loc[SPECS, MODELS]
ax = tab.plot.bar(figsize=(10, 4), rot=0)
ax.axhline(0.351, color='k', ls='--', lw=1, label='rule baseline (0.351)')
ax.set_ylabel('macro-F1 (leave-one-recording-out)'); ax.legend(ncol=3, fontsize=8)
plt.title('Classic models x feature sets'); plt.tight_layout(); plt.show()

In [ ]:
# 6. 예측 저장 (오류 분석용, 드라이브)
from google.colab import drive
drive.mount('/content/drive')
pd.DataFrame({'y': y, 'rec': rec, **preds}).to_csv('/content/drive/MyDrive/ML-imu_B_preds.csv', index=False)
print('saved')

In [ ]:
# 7. results.csv에 넣을 줄 (출력된 글자를 전부 복사해서 Claude에게 붙여넣기)
from datetime import date
out = res.assign(date=date.today().isoformat(), owner='B', setting='8class_' + res.features,
                 fold='all', auroc='', fpr95='', note='')
cols = ['date','owner','model','setting','fold','macro_f1','accuracy','mode_macro_f1','auroc','fpr95','note']
print(out[cols].round(4).to_csv(index=False, header=False))